# AI应用基础 · 工单M2.1《工业焊接缺陷检测系统》 — 教师参考版
第6周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

subprocess.run([sys.executable, "数据与工具/generate_data.py", "m2img"], check=True)
print("数据已生成: data/")

工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础


M2.1 -> data/weld_images 240 张
完成。数据输出目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础/data
数据已生成: data/


## 参考实现 `m21`（含依赖导入）

In [2]:
import csv
import json
import sys
import numpy as np
import numpy as np  # 通用科学计算


def m21():
    import torch
    import torch.nn as nn

    with open("data/weld_images/index.json", encoding="utf-8") as f:
        meta = json.load(f)
    X = np.stack([np.load("data/weld_images/" + m["file"]) for m in meta])
    y = np.array([["normal", "porosity", "slag", "crack"].index(m["label"])
                  for m in meta])
    X = (X.astype(np.float32) / 255.0)[:, None]
    idx = np.random.RandomState(42).permutation(len(X))
    X, y = X[idx], y[idx]
    n = int(len(X) * 0.8)
    Xtr, ytr = torch.tensor(X[:n]), torch.tensor(y[:n])
    Xte, yte = torch.tensor(X[n:]), torch.tensor(y[n:])

    class TinyCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 8, 3)
            self.pool = nn.MaxPool2d(2, 2)
            self.conv2 = nn.Conv2d(8, 16, 3)
            self.fc1 = nn.Linear(16 * 14 * 14, 64)
            self.fc2 = nn.Linear(64, 4)

        def forward(self, x):
            x = self.pool(torch.relu(self.conv1(x)))
            x = self.pool(torch.relu(self.conv2(x)))
            x = x.flatten(1)
            return self.fc2(torch.relu(self.fc1(x)))

    torch.manual_seed(42)
    model = TinyCNN()
    opt = torch.optim.Adam(model.parameters(), lr=1.5e-3)
    lossf = nn.CrossEntropyLoss()
    for ep in range(6):
        model.train()
        perm = torch.randperm(len(Xtr))
        tot = 0.0
        for i in range(0, len(Xtr), 32):
            b = perm[i:i + 32]
            opt.zero_grad()
            loss = lossf(model(Xtr[b]), ytr[b])
            loss.backward()
            opt.step()
            tot += loss.item() * len(b)
        model.eval()
        with torch.no_grad():
            acc = (model(Xte).argmax(1) == yte).float().mean().item()
        print(f"epoch{ep + 1} loss={tot / len(Xtr):.4f} 测试acc={acc:.3f}")
    print("形状追踪: (N,1,64,64)→conv1(N,8,62,62)→pool(31)→conv2(N,16,29,29)"
          "→pool(14)→flatten(N,3136)→fc→(N,4)")


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `2` | TODO-2: 小批量训练(batch=32): 前向→loss→反向→step, 累计打印epoch loss |
|  | （在此处编写你的代码——依据上方注释/文档字符串） |
| `3` | TODO-3: 任取2张测试图, 输出预测类别与softmax置信度 TODO-4: (优秀)数据增强: |
|  | 随机水平翻转后再训练, 对比准确率 TODO-5: (优秀)绘制loss曲线与混淆矩阵(matplotlib) |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 执行与验证

In [3]:
m21()

epoch1 loss=1.3993 测试acc=0.208
epoch2 loss=1.3748 测试acc=0.521


epoch3 loss=1.3492 测试acc=0.500
epoch4 loss=1.2775 测试acc=0.854


epoch5 loss=1.1797 测试acc=0.854
epoch6 loss=1.0119 测试acc=0.875
形状追踪: (N,1,64,64)→conv1(N,8,62,62)→pool(31)→conv2(N,16,29,29)→pool(14)→flatten(N,3136)→fc→(N,4)
